# Focused 4-Class Intrusion Detection (CIC-IDS-2017 + BCCC-CIC-IDS-2017) — Colab + Google Drive

This notebook runs a focused 4-class experiment on **both datasets** and trains multiple models.

## Classes kept
- BENIGN
- BRUTE_FORCE (FTP-Patator, SSH-Patator, Brute Force -*)
- XSS
- SQL_INJECTION

## Models
- LightGBM (multiclass)
- Decision Tree (entropy; ID3-like)
- k-NN
- Naive Bayes (GaussianNB)
- Random Forest
- Linear SVM (calibrated)
- ANN (MLP)
- 1D CNN

## Metrics in tables
- Accuracy
- Macro Precision
- Macro Recall
- Macro F1
- Weighted F1
- Macro ROC-AUC (OvR)

## Output saving (your request)
**All outputs** (tables + reports + plots) are saved into a single folder in Google Drive:
- `tables/` → CSVs (per-model tables + final combined)
- `reports/` → text classification reports
- `figures/` → PNGs (confusion matrices + learning curves + class balance plots)


## Cell 1 — Install dependencies (Colab)

In [ ]:
!pip -q install lightgbm joblib tensorflow scikit-learn

## Cell 2 — Imports & global settings

In [ ]:
import os, time, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

import lightgbm as lgb

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 3 — Mount Google Drive & set dataset paths

In [ ]:
# Update if your folder differs
DRIVE_ROOT = "/content/drive/MyDrive/datasets"

DATASET_PATHS = {
    "CIC-IDS- 2017": f"{DRIVE_ROOT}/CIC-IDS- 2017",
    "BCCC-CIC-IDS-2017": f"{DRIVE_ROOT}/BCCC-CIC-IDS-2017",
}

DATASETS_TO_RUN = ["CIC-IDS- 2017", "BCCC-CIC-IDS-2017"]

for k in DATASETS_TO_RUN:
    print(k, "->", DATASET_PATHS[k])


## Cell 4 — Output folder (ALL outputs saved here)

This cell creates a single experiment folder in your Drive and subfolders for:
- tables
- reports
- figures

Change `RUN_NAME` to version your experiments.


In [ ]:
from datetime import datetime

SAVE_ALL_OUTPUTS = True

RUN_NAME = "focused4class_cic_vs_bccc"
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

OUTPUT_ROOT = f"{DRIVE_ROOT}/outputs/{RUN_NAME}_{TIMESTAMP}"
TABLE_DIR  = f"{OUTPUT_ROOT}/tables"
REPORT_DIR = f"{OUTPUT_ROOT}/reports"
FIG_DIR    = f"{OUTPUT_ROOT}/figures"
MACRO_DIR  = f"{OUTPUT_ROOT}/internal_macros"

if SAVE_ALL_OUTPUTS:
    os.makedirs(TABLE_DIR, exist_ok=True)
    os.makedirs(REPORT_DIR, exist_ok=True)
    os.makedirs(FIG_DIR, exist_ok=True)
    os.makedirs(MACRO_DIR, exist_ok=True)

print("OUTPUT_ROOT:", OUTPUT_ROOT)


## Cell 5 — (Optional) Verify files in each dataset folder

In [ ]:
for key in DATASETS_TO_RUN:
    path = DATASET_PATHS[key]
    print(f"\n{key} → {path}")
    if not os.path.isdir(path):
        print("  Folder not found")
        continue
    files = sorted([f for f in os.listdir(path) if f.lower().endswith('.csv')])
    print(f"  Found {len(files)} CSV files")
    for f in files[:20]:
        print("   -", f)
    if len(files) > 20:
        print("   ...")


## Cell 6 — Configuration (splits, sampling, hyperparameters)

Split:
- train = 70%
- val = 10%
- test = 20%

Sampling applies to train/val only (test unchanged).


In [ ]:
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.10
TEST_FRAC  = 0.20
assert abs(TRAIN_FRAC + VAL_FRAC + TEST_FRAC - 1.0) < 1e-9

# Sampling caps (set None to disable)
MAX_TRAIN_SAMPLES = 300_000
MAX_VAL_SAMPLES   = 100_000

# Models
RUN_LGBM = True
RUN_DT   = True
RUN_KNN  = True
RUN_NB   = True
RUN_RF   = True
RUN_SVM  = True
RUN_ANN  = True
RUN_CNN  = True

# kNN
KNN_K = 15

# Random Forest
RF_TREES = 200
RF_MAX_DEPTH = None  # e.g., 20 to speed up

# ANN/CNN
EPOCHS = 16
BATCH_SIZE = 4096

# LightGBM
LGBM_N_ESTIMATORS = 5000
LGBM_LR = 0.05
LGBM_NUM_LEAVES = 63
EARLY_STOPPING_ROUNDS = 100


## Cell 7 — Utility: safe filenames + save helper

In [ ]:
def safe_name(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"[^a-z0-9\-\_]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def save_fig(filename: str):
    if not SAVE_ALL_OUTPUTS:
        return
    fp = os.path.join(FIG_DIR, filename)
    plt.savefig(fp, dpi=200, bbox_inches="tight")
    print("Saved figure:", fp)

def save_text(filename: str, text: str):
    if not SAVE_ALL_OUTPUTS:
        return
    fp = os.path.join(REPORT_DIR, filename)
    with open(fp, "w", encoding="utf-8") as f:
        f.write(text)
    print("Saved report:", fp)

def save_csv(filename: str, df: pd.DataFrame):
    if not SAVE_ALL_OUTPUTS:
        return
    fp = os.path.join(TABLE_DIR, filename)
    df.to_csv(fp, index=False)
    print("Saved table:", fp)

def save_macro_csv(filename: str, df: pd.DataFrame):
    if not SAVE_ALL_OUTPUTS:
        return
    fp = os.path.join(MACRO_DIR, filename)
    df.to_csv(fp, index=False)
    print("Saved macro table:", fp)



In [ ]:
def audit_labels_and_save(df: pd.DataFrame, dataset_key: str):
    """
    Robust audit:
    - strips column names
    - detects label column (works for Label / label)
    - saves raw label counts, XSS/SQL subsets, and raw->mapped counts
    """
    df_a = df.copy()
    df_a.columns = df_a.columns.str.strip()

    label_col = detect_label_column(df_a)  # always re-detect safely
    raw = df_a[label_col].astype(str).str.strip()

    # 1) Raw label counts
    raw_counts = raw.value_counts().reset_index()
    raw_counts.columns = ["raw_label", "count"]
    raw_counts["dataset"] = dataset_key
    save_csv(f"raw_label_counts_{safe_name(dataset_key)}.csv", raw_counts)

    # 2) XSS-containing raw labels
    xss_counts = raw[raw.str.contains("XSS", case=False, na=False)].value_counts().reset_index()
    xss_counts.columns = ["raw_label", "count"]
    xss_counts["dataset"] = dataset_key
    save_csv(f"raw_labels_contains_xss_{safe_name(dataset_key)}.csv", xss_counts)

    # 3) SQL-containing raw labels
    sql_counts = raw[raw.str.contains("SQL", case=False, na=False)].value_counts().reset_index()
    sql_counts.columns = ["raw_label", "count"]
    sql_counts["dataset"] = dataset_key
    save_csv(f"raw_labels_contains_sql_{safe_name(dataset_key)}.csv", sql_counts)

    # 4) Raw → mapped counts
    mapped = raw.apply(normalize_attack_label)
    map_df = pd.DataFrame({"raw_label": raw, "mapped_label": mapped})
    map_counts = (
        map_df.groupby(["raw_label", "mapped_label"])
             .size()
             .reset_index(name="count")
             .sort_values("count", ascending=False)
    )
    map_counts["dataset"] = dataset_key
    save_csv(f"raw_to_mapped_counts_{safe_name(dataset_key)}.csv", map_counts)

    print(f"\n=== Label audit for {dataset_key} (label_col='{label_col}') ===")
    display(raw_counts.head(20))
    display(xss_counts.head(20))
    display(sql_counts.head(20))
    display(map_counts.head(30))


## Cell 8 — Load all CSVs from a folder

In [ ]:
def load_dataset(folder_path: str) -> pd.DataFrame:
    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f"Folder not found: {folder_path}")
    csv_files = sorted([f for f in os.listdir(folder_path) if f.lower().endswith('.csv')])
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in: {folder_path}")
    dfs = []
    for f in csv_files:
        fp = os.path.join(folder_path, f)
        try:
            dfs.append(pd.read_csv(fp))
        except Exception as e:
            print(f"  Skipping {f} (read error): {e}")
    if not dfs:
        raise RuntimeError("All CSV files failed to load.")
    return pd.concat(dfs, ignore_index=True)


## Cell 9 — Focused 4-class label mapping + preprocessing

Mapping rules:
- BENIGN variants → `BENIGN`
- `FTP-Patator`, `SSH-Patator`, `Brute Force -Web`, `Brute Force -XSS`, and anything containing `PATATOR` → `BRUTE_FORCE`
- `XSS` (pure label) → `XSS`
- `SQL Injection` → `SQL_INJECTION`
All other labels are excluded.

`Brute Force -XSS` is treated as **BRUTE_FORCE** by rule order.


In [ ]:
BENIGN_TOKENS = {"BENIGN", "BENGIN", "NORMAL", "benign", "normal", "Normal"}
TARGET_CLASSES = ["BENIGN", "BRUTE_FORCE", "XSS", "SQL_INJECTION"]

def detect_label_column(df: pd.DataFrame) -> str:
    colmap = {c.strip().lower(): c for c in df.columns}
    for cand in ["label", "attack", "attack_cat", "class"]:
        if cand in colmap:
            return colmap[cand]
    raise ValueError("Could not find label column (expected Label/label).")

def normalize_attack_label(raw):
    if raw is None:
        return None

    su = str(raw).strip().upper()

    # Normalize separators: "_" "-" "/" -> spaces
    norm = re.sub(r"[_\-/]+", " ", su)
    norm = re.sub(r"\s+", " ", norm).strip()

    # BENIGN
    if norm in {b.upper() for b in BENIGN_TOKENS}:
        return "BENIGN"

    # SQL Injection
    if "SQL" in norm and "INJECTION" in norm:
        return "SQL_INJECTION"

    # Brute-force family
    if ("BRUTE" in norm and "FORCE" in norm) or ("PATATOR" in norm):
        return "BRUTE_FORCE"

    # XSS (this will catch Web_XSS)
    if "XSS" in norm:
        return "XSS"

    return None


def clean_prepare_filtered_multiclass(df: pd.DataFrame):
    df = df.copy()
    df.columns = df.columns.str.strip()
    df = df.drop_duplicates()

    label_col = detect_label_column(df)

    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) > 0:
        df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)

    df = df.dropna()

    y_norm = df[label_col].apply(normalize_attack_label)
    keep = y_norm.notna()
    df = df.loc[keep].copy()
    y_norm = y_norm.loc[keep].astype(str)

    X = df.drop(columns=[label_col, "is_attack"], errors="ignore")
    X = X.select_dtypes(include=[np.number]).copy()

    if X.shape[1] == 0:
        raise ValueError("No numeric features found after preprocessing.")
    if y_norm.nunique() < 2:
        raise ValueError("Not enough classes after filtering. Check mapping or dataset content.")

    return X, y_norm, label_col


## Cell 10 — Load both datasets + class balance outputs (saved)

This cell:
- loads and filters each dataset to the 4 classes
- displays the class-count table
- saves the class-count table to CSV
- plots and saves the class-count bar chart


In [ ]:
datasets = {}
summary_rows = []

def plot_class_counts(y, title, fig_name):
    vc = y.value_counts().reindex(TARGET_CLASSES).fillna(0).astype(int)
    plt.figure(figsize=(7,4))
    vc.plot(kind="bar")
    plt.title(title)
    plt.xlabel("Class")
    plt.ylabel("Count")
    plt.tight_layout()
    save_fig(fig_name)
    plt.show()
    plt.close()

for key in DATASETS_TO_RUN:
    print("\n" + "="*80)
    print("Loading:", key)
    df = load_dataset(DATASET_PATHS[key])
    df.columns = df.columns.str.strip()
    X, y_str, label_col = clean_prepare_filtered_multiclass(df)

    le = LabelEncoder()
    y_int = le.fit_transform(y_str)
    class_names = list(le.classes_)

    datasets[key] = {
        "X": X,
        "y_str": y_str,
        "y_int": y_int,
        "label_encoder": le,
        "class_names": class_names,
        "label_col": label_col,
    }

    counts = y_str.value_counts().reindex(TARGET_CLASSES).fillna(0).astype(int).to_frame("count")
    display(counts)

    # Save counts table
    save_csv(f"class_balance_{safe_name(key)}.csv", counts.reset_index().rename(columns={"index":"class"}))

    audit_labels_and_save(df, dataset_key=key)


    # Save plot
    plot_class_counts(y_str, f"{key}: Class counts (filtered 4-class)",
                      fig_name=f"class_balance_{safe_name(key)}.png")

    summary_rows.append({
        "dataset": key,
        "rows_after_filter": len(y_str),
        "n_numeric_features": X.shape[1],
        "label_column": label_col,
        "classes_present": ", ".join(class_names)
    })

summary_df = pd.DataFrame(summary_rows).sort_values("dataset")
display(summary_df)
save_csv("dataset_summary.csv", summary_df)


## Cell 11 — Train/Val/Test split (70/10/20) + sampling helpers

In [ ]:
def make_splits_multiclass(X, y_int):
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y_int, test_size=TEST_FRAC, random_state=RANDOM_STATE, stratify=y_int
    )
    val_from_temp = VAL_FRAC / (TRAIN_FRAC + VAL_FRAC)  # 0.125
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=val_from_temp, random_state=RANDOM_STATE, stratify=y_temp
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

def maybe_sample(X, y, max_n):
    if max_n is None or len(y) <= max_n:
        return X, y
    rng = np.random.RandomState(RANDOM_STATE)
    idx = rng.choice(len(y), size=max_n, replace=False)
    if isinstance(X, pd.DataFrame):
        return X.iloc[idx], y[idx]
    return X[idx], y[idx]

def sample_splits(X_train, y_train, X_val, y_val):
    X_train_s, y_train_s = maybe_sample(X_train, y_train, MAX_TRAIN_SAMPLES)
    X_val_s, y_val_s     = maybe_sample(X_val, y_val, MAX_VAL_SAMPLES)
    return X_train_s, y_train_s, X_val_s, y_val_s

def make_scaled(X_train, X_val, X_test):
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s   = scaler.transform(X_val)
    X_test_s  = scaler.transform(X_test)
    return X_train_s, X_val_s, X_test_s, scaler


## Cell 12 — Metrics (includes Macro ROC-AUC)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def compute_metrics_main_and_macro(y_true, y_pred, y_proba=None):
    #  Main metrics (what you display in your primary tables)
    main = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "weighted_precision": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
        "weighted_recall": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
    }

    #  Macro metrics (computed internally, saved elsewhere)
    macro = {
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
    }

    # Macro ROC-AUC requires probabilities
    if y_proba is not None:
        macro["macro_roc_auc"] = float(
            roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro")
        )
    else:
        macro["macro_roc_auc"] = np.nan

    return main, macro



from sklearn.metrics import classification_report

def per_class_table(y_true, y_pred, class_names, dataset_name, model_name):
    """Return long-form per-class metrics table."""
    rep_dict = classification_report(
        y_true, y_pred,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )
    rows = []
    for cls in class_names:
        if cls in rep_dict:
            rows.append({
                "dataset": dataset_name,
                "model": model_name,
                "class": cls,
                "precision": rep_dict[cls]["precision"],
                "recall": rep_dict[cls]["recall"],
                "f1": rep_dict[cls]["f1-score"],
                "support": rep_dict[cls]["support"],
            })
    return pd.DataFrame(rows)


## Cell 13 — Plot helpers (confusion + learning curves) with saving

In [ ]:
def plot_confusion(cm, class_names, title, fig_name):
    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    plt.xticks(range(len(class_names)), class_names, rotation=45, ha="right")
    plt.yticks(range(len(class_names)), class_names)
    thresh = cm.max() / 2.0 if cm.size else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center",
                     color="white" if cm[i, j] > thresh else "black")
    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.tight_layout()
    save_fig(fig_name)
    plt.show()
    plt.close()

def plot_keras_learning(history, title, fig_prefix):
    plt.figure()
    plt.plot(history.history.get("accuracy", []), label="train acc")
    plt.plot(history.history.get("val_accuracy", []), label="val acc")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.tight_layout()
    save_fig(f"{fig_prefix}_acc.png")
    plt.show()
    plt.close()

    plt.figure()
    plt.plot(history.history.get("loss", []), label="train loss")
    plt.plot(history.history.get("val_loss", []), label="val loss")
    plt.title(title + " (loss)")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    save_fig(f"{fig_prefix}_loss.png")
    plt.show()
    plt.close()


## Cell 14 — Model builders (sklearn) and LightGBM trainer

In [ ]:
def build_dt():
    return DecisionTreeClassifier(criterion="entropy", random_state=RANDOM_STATE)

def build_knn():
    return KNeighborsClassifier(n_neighbors=KNN_K, n_jobs=-1)

def build_nb():
    return GaussianNB()

def build_rf():
    return RandomForestClassifier(
        n_estimators=RF_TREES,
        max_depth=RF_MAX_DEPTH,
        n_jobs=-1,
        random_state=RANDOM_STATE
    )

def build_svm():
    base = LinearSVC(random_state=RANDOM_STATE)
    return CalibratedClassifierCV(base, method="sigmoid", cv=3)

def train_lgbm(X_train, y_train, X_val, y_val, n_classes):
    model = lgb.LGBMClassifier(
        objective="multiclass",
        num_class=n_classes,
        n_estimators=LGBM_N_ESTIMATORS,
        learning_rate=LGBM_LR,
        num_leaves=LGBM_NUM_LEAVES,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="multi_logloss",
        callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=True)]
    )
    return model


## Cell 15 — ANN (MLP) and 1D CNN (Keras)

In [ ]:
def build_ann(input_dim, n_classes):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(n_classes, activation="softmax"),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

def build_cnn(input_dim, n_classes):
    model = keras.Sequential([
        layers.Input(shape=(input_dim, 1)),
        layers.Conv1D(64, 3, activation="relu", padding="same"),
        layers.MaxPooling1D(2),
        layers.Conv1D(128, 3, activation="relu", padding="same"),
        layers.GlobalMaxPooling1D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(n_classes, activation="softmax"),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


## Cell 16 — Train/evaluate models (saves everything)

What gets saved:
- Per-dataset classification report: `reports/<dataset>__<model>.txt`
- Confusion matrix figure: `figures/<dataset>__<model>__confusion.png`
- ANN/CNN learning curves: `figures/<dataset>__<model>__learning_acc.png` and `..._loss.png`
- Per-model comparison table across datasets: `tables/per_model_<model>.csv`
- Final combined table: `tables/final_combined.csv`


In [ ]:
def run_sklearn_model(model, Xtr, ytr, Xte, yte, class_names, dataset_name, model_name):
    t0 = time.time()
    model.fit(Xtr, ytr)
    fit_s = time.time() - t0

    t0 = time.time()
    y_proba = model.predict_proba(Xte)
    pred_s = time.time() - t0

    y_pred = np.argmax(y_proba, axis=1)
    main_metrics, macro_metrics = compute_metrics_main_and_macro(yte, y_pred, y_proba)

    # Per-class metrics table (saved later)
    per_cls_df = per_class_table(yte, y_pred, class_names, dataset_name, model_name)

    # Save report
    rep = classification_report(yte, y_pred, target_names=class_names, digits=4, zero_division=0)
    save_text(f"{safe_name(dataset_name)}__{safe_name(model_name)}.txt", rep)

    # Plot + save confusion
    cm = confusion_matrix(yte, y_pred)
    plot_confusion(cm, class_names,
                   title=f"{dataset_name} — {model_name} Confusion",
                   fig_name=f"{safe_name(dataset_name)}__{safe_name(model_name)}__confusion.png")

    return main_metrics, macro_metrics, per_cls_df, fit_s, pred_s

def run_lgbm_model(Xtr, ytr, Xval, yval, Xte, yte, class_names, dataset_name, model_name):
    t0 = time.time()
    model = train_lgbm(Xtr, ytr, Xval, yval, n_classes=len(class_names))
    fit_s = time.time() - t0

    t0 = time.time()
    y_proba = model.predict_proba(Xte)
    pred_s = time.time() - t0

    y_pred = np.argmax(y_proba, axis=1)
    main_metrics, macro_metrics = compute_metrics_main_and_macro(yte, y_pred, y_proba)

    # Per-class metrics table (saved later)
    per_cls_df = per_class_table(yte, y_pred, class_names, dataset_name, model_name)


    rep = classification_report(yte, y_pred, target_names=class_names, digits=4, zero_division=0)
    save_text(f"{safe_name(dataset_name)}__{safe_name(model_name)}.txt", rep)

    cm = confusion_matrix(yte, y_pred)
    plot_confusion(cm, class_names,
                   title=f"{dataset_name} — {model_name} Confusion",
                   fig_name=f"{safe_name(dataset_name)}__{safe_name(model_name)}__confusion.png")
    return main_metrics, macro_metrics, per_cls_df, fit_s, pred_s


def run_keras_model(kind, build_fn, Xtr, ytr, Xval, yval, Xte, yte, class_names, dataset_name, model_name, epochs, batch_size):
    if kind == "CNN":
        model = build_fn(Xtr.shape[1], len(class_names))
        Xtr_in = Xtr[..., np.newaxis]
        Xval_in = Xval[..., np.newaxis]
        Xte_in = Xte[..., np.newaxis]
    else:
        model = build_fn(Xtr.shape[1], len(class_names))
        Xtr_in, Xval_in, Xte_in = Xtr, Xval, Xte

    t0 = time.time()
    history = model.fit(
        Xtr_in, ytr,
        validation_data=(Xval_in, yval),
        epochs=epochs,
        batch_size=batch_size,
        verbose=1
    )
    fit_s = time.time() - t0

    t0 = time.time()
    y_proba = model.predict(Xte_in, batch_size=batch_size, verbose=0)
    pred_s = time.time() - t0

    y_pred = np.argmax(y_proba, axis=1)
    main_metrics, macro_metrics = compute_metrics_main_and_macro(yte, y_pred, y_proba)

    # Per-class metrics table (saved later)
    per_cls_df = per_class_table(yte, y_pred, class_names, dataset_name, model_name)

    rep = classification_report(yte, y_pred, target_names=class_names, digits=4, zero_division=0)
    save_text(f"{safe_name(dataset_name)}__{safe_name(model_name)}.txt", rep)

    plot_keras_learning(history,
                        title=f"{dataset_name} — {model_name} Learning Curves",
                        fig_prefix=f"{safe_name(dataset_name)}__{safe_name(model_name)}__learning")

    cm = confusion_matrix(yte, y_pred)
    plot_confusion(cm, class_names,
                   title=f"{dataset_name} — {model_name} Confusion",
                   fig_name=f"{safe_name(dataset_name)}__{safe_name(model_name)}__confusion.png")

    return main_metrics, macro_metrics, per_cls_df, fit_s, pred_s

# Prepare splits once per dataset so all models share same test set
prepared = {}
for ds in DATASETS_TO_RUN:
    X = datasets[ds]["X"]
    y = datasets[ds]["y_int"]
    le = datasets[ds]["label_encoder"]
    class_names = list(le.classes_)

    X_train, X_val, X_test, y_train, y_val, y_test = make_splits_multiclass(X, y)
    X_train_s, y_train_s, X_val_s, y_val_s = sample_splits(X_train, y_train, X_val, y_val)

    Xtr_sc, Xval_sc, Xte_sc, scaler = make_scaled(X_train_s, X_val_s, X_test)

    prepared[ds] = {
        "Xtr": Xtr_sc, "ytr": y_train_s,
        "Xval": Xval_sc, "yval": y_val_s,
        "Xte": Xte_sc, "yte": y_test,
        "class_names": class_names
    }

model_specs = []
if RUN_LGBM: model_specs.append(("LightGBM", "lgbm"))
if RUN_DT:   model_specs.append(("DT(entropy)", "dt"))
if RUN_KNN:  model_specs.append((f"kNN(k={KNN_K})", "knn"))
if RUN_NB:   model_specs.append(("GaussianNB", "nb"))
if RUN_RF:   model_specs.append((f"RF(trees={RF_TREES})", "rf"))
if RUN_SVM:  model_specs.append(("LinearSVM(cal)", "svm"))
if RUN_ANN:  model_specs.append(("ANN(MLP)", "ann"))
if RUN_CNN:  model_specs.append(("CNN(1D)", "cnn"))

def build_model(kind):
    if kind == "dt": return build_dt()
    if kind == "knn": return build_knn()
    if kind == "nb": return build_nb()
    if kind == "rf": return build_rf()
    if kind == "svm": return build_svm()
    raise ValueError(kind)

# -----------------------------
# MAIN result store (weighted metrics only)
# -----------------------------
all_rows_main = []
all_rows_macro = []
all_rows_per_class = []

for model_name, kind in model_specs:
    per_model_rows_main = []
    per_model_rows_macro = []

    for ds in DATASETS_TO_RUN:
        Xtr = prepared[ds]["Xtr"]; ytr = prepared[ds]["ytr"]
        Xval = prepared[ds]["Xval"]; yval = prepared[ds]["yval"]
        Xte = prepared[ds]["Xte"]; yte = prepared[ds]["yte"]
        class_names = prepared[ds]["class_names"]

        if kind == "lgbm":
            main_metrics, macro_metrics, per_cls_df, _, _ = run_lgbm_model(
                Xtr, ytr, Xval, yval, Xte, yte, class_names, ds, model_name
            )
        elif kind in {"dt","knn","nb","rf","svm"}:
            model = build_model(kind)
            main_metrics, macro_metrics, per_cls_df, _, _ = run_sklearn_model(
                model, Xtr, ytr, Xte, yte, class_names, ds, model_name
            )
        elif kind == "ann":
            main_metrics, macro_metrics, per_cls_df, _, _ = run_keras_model(
                "ANN", build_ann, Xtr, ytr, Xval, yval, Xte, yte,
                class_names, ds, model_name, EPOCHS, BATCH_SIZE
            )
        elif kind == "cnn":
            main_metrics, macro_metrics, per_cls_df, _, _ = run_keras_model(
                "CNN", build_cnn, Xtr, ytr, Xval, yval, Xte, yte,
                class_names, ds, model_name, EPOCHS, BATCH_SIZE
            )
        else:
            raise ValueError(kind)

        row_main = {"dataset": ds, "model": model_name, **main_metrics}
        row_macro = {"dataset": ds, "model": model_name, **macro_metrics}

        per_model_rows_main.append(row_main)
        per_model_rows_macro.append(row_macro)

        all_rows_main.append(row_main)
        all_rows_macro.append(row_macro)
        all_rows_per_class.append(per_cls_df)

    # Save per-model tables (main + macros)
    per_model_df_main = pd.DataFrame(per_model_rows_main).sort_values("dataset")
    per_model_df_macro = pd.DataFrame(per_model_rows_macro).sort_values("dataset")

    print("\n" + "-" * 90)
    print(f"PER-MODEL RESULTS (Weighted metrics): {model_name}")
    display(per_model_df_main[[
        "dataset","accuracy","weighted_precision","weighted_recall","weighted_f1"
    ]])

    save_csv(f"per_model_{safe_name(model_name)}.csv", per_model_df_main)
    save_macro_csv(f"per_model_{safe_name(model_name)}__macros.csv", per_model_df_macro)

# Final combined tables
results_main = pd.DataFrame(all_rows_main)
results_macro = pd.DataFrame(all_rows_macro)

print("\n" + "=" * 90)
print("FINAL COMBINED RESULTS (Weighted metrics)")
display(results_main[[
    "dataset","model","accuracy","weighted_precision","weighted_recall","weighted_f1"
]].sort_values(["dataset","weighted_f1"], ascending=[True, False]))

save_csv("final_combined.csv", results_main)
save_macro_csv("final_combined__macros.csv", results_macro)

# Final per-class table (ALL models × BOTH datasets)
per_class_final = pd.concat(all_rows_per_class, ignore_index=True)
print("\n" + "=" * 90)
print("FINAL PER-CLASS METRICS TABLE")
display(per_class_final.sort_values(["dataset","model","class"]))
save_csv("per_class_final.csv", per_class_final)



print("All outputs saved under:", OUTPUT_ROOT)


In [ ]:
print(results_main.columns.tolist())


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

per_class_final = pd.read_csv(f"{TABLE_DIR}/per_class_final.csv")
os.makedirs(FIG_DIR, exist_ok=True)

SHOW_PLOTS = True

CLASS_ORDER = ["BENIGN", "BRUTE_FORCE", "XSS", "SQL_INJECTION"]

def plot_dataset_one_chart(dataset_name: str):
    df = per_class_final[per_class_final["dataset"] == dataset_name].copy()
    if df.empty:
        print("No rows for dataset:", dataset_name)
        return

    # Order classes + models consistently
    df["class"] = pd.Categorical(df["class"], categories=CLASS_ORDER, ordered=True)
    models = sorted(df["model"].unique())

    # Create one x category per (model, class)
    df = df.sort_values(["model", "class"])
    df["xcat"] = df["model"].astype(str) + " | " + df["class"].astype(str)

    xcats = df["xcat"].tolist()
    x = np.arange(len(xcats))
    width = 0.27

    plt.figure(figsize=(max(14, 0.55 * len(xcats)), 5))
    plt.bar(x - width, df["precision"].values, width, label="Precision")
    plt.bar(x,         df["recall"].values,    width, label="Recall")
    plt.bar(x + width, df["f1"].values,        width, label="F1")

    plt.title(f"{dataset_name} — Per-class Precision / Recall / F1 across Models")
    plt.ylabel("Score")
    plt.ylim(0, 1)
    plt.grid(axis="y", alpha=0.3)

    plt.xticks(x, xcats, rotation=90, ha="center", fontsize=8)
    plt.legend(ncol=3)
    plt.tight_layout()

    out_path = os.path.join(FIG_DIR, f"{safe_name(dataset_name)}__ONECHART_models_classes_PRF.png")
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    plt.close()

    print("Saved:", out_path)

# Exactly two charts (one per dataset)
for ds in sorted(per_class_final["dataset"].unique()):
    plot_dataset_one_chart(ds)
